# Audio a texto con modelos abiertos · tutorial corto

Todo corre **en tu computadora**: el audio de tus participantes no sale a ninguna API.

1. Whisper (OpenAI, MIT) en CPU: entrevista en español y en inglés
2. Qwen3-ASR (Alibaba, China, Apache 2.0) en GPU, con nombres propios como contexto
3. Alucinaciones: Whisper "oye" frases en el silencio
4. Quechua de Huaylas con MMS (Meta)
5. ¿Quién habla? Separar hablantes

```bash
uv sync --group audio
uv run python sessions/13-audio-a-texto/datos.py
```

In [1]:
import os, sys, time, warnings
from pathlib import Path
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
from transformers.utils import logging; logging.set_verbosity_error(); logging.disable_progress_bar()

SESION = Path.cwd().parent if Path.cwd().name == "lecture" else Path.cwd()
sys.path.insert(0, str(SESION))
import datos, voz
from voz import SR, Transcriptor, cargar_audio, diarizar, a_texto

datos.bajar_commons()
DATA = SESION / "data" / "entrevistas"
print("Dispositivo:", voz.dispositivo())

Dispositivo: mps


## 1 · Whisper en CPU (faster-whisper): funciona en cualquier laptop

In [2]:
from faster_whisper import WhisperModel
whisper = WhisperModel("mobiuslabsgmbh/faster-whisper-large-v3-turbo", device="cpu", compute_type="int8", cpu_threads=8)

x = cargar_audio(DATA / "agustino_1.ogg")[:45 * SR]       # entrevista en El Agustino, Lima
t0 = time.perf_counter()
segmentos, info = whisper.transcribe(x, beam_size=5, vad_filter=True)
for s in segmentos:
    print(f"[{s.start:5.1f} → {s.end:5.1f}] {s.text.strip()}")
print(f"\nIdioma detectado: {info.language} · {len(x)/SR:.0f} s de audio en {time.perf_counter()-t0:.0f} s")

[  0.3 →   7.7] Buenos días, aquí son las 10 y 28 y estoy con una señora que va a darme su opinión acerca del partido de hoy.
[  8.4 →   9.7] ¿Puede decirme su nombre por favor?
[ 10.3 →  11.4] Elena Durán.
[ 12.2 →  17.6] Mucho gusto, yo soy Jorge, soy alumno de la Universidad Nacional Mayor de San Marcos
[ 17.6 →  21.5] y le voy a hacer estas preguntas acerca del partido de hoy.
[ 22.5 →  26.0] ¿Qué expectativas tiene acerca de este partido?
[ 26.4 →  29.2] ¿Quién piensa que va a atacar más o defender?


[ 30.3 →  33.9] Bueno, esperamos que Perú ganemos, ¿no?
[ 35.6 →  38.8] Espero que Padula se eche goles.

Idioma detectado: es · 45 s de audio en 8 s


In [3]:
x_en = cargar_audio(DATA / "historia_oral_glines.ogg")[:40 * SR]   # historia oral en inglés
segmentos, info = whisper.transcribe(x_en, beam_size=5, vad_filter=True)
print(info.language, "→", " ".join(s.text.strip() for s in segmentos))

en → We just took a brief break for a phone call. And I just asked, is there anything else you'd like to share about Wilton Campus School? Well, the only thing was that it really was, it wasn't perfect. I mean, we had problems. We didn't do everything perfect. You know, there were lots of things that I'd do differently. But remember, we were an experimental school. We were trying new ideas. We were trying new things. And that's what Minnesota's lost. We don't have a school like Wilton now, public school, supported by the state, by the college.


## 2 · Qwen3-ASR (Alibaba): más rápido en GPU y acepta contexto

In [4]:
qwen = Transcriptor("qwen3-asr-0.6b")
x2 = cargar_audio(DATA / "agustino_2.ogg")
nombres = x2[10 * SR:13 * SR]
print("sin contexto:", qwen.transcribir(nombres, "es"))
print("con contexto:", qwen.transcribir(nombres, "es", contexto="Isabel Vásquez, Elsa Barrientos"))

sin contexto: ¿Sabes? Sabes el básquet. Sabes a barrientos.
con contexto: Isabel Vásquez Elsa Barrientos


## 3 · Alucinaciones: 10 segundos de silencio

In [5]:
import numpy as np
silencio = np.zeros(10 * SR, dtype=np.float32)
print("Whisper sin VAD:", [s.text for s in whisper.transcribe(silencio, language="es")[0]])
print("Whisper con VAD:", [s.text for s in whisper.transcribe(silencio, language="es", vad_filter=True)[0]])

Whisper sin VAD: [' Gracias por ver el video']
Whisper con VAD: []


Whisper aprendió de subtítulos de YouTube: en el silencio escribe «Gracias por ver el video». **Activa siempre el VAD.**

## 4 · Quechua de Huaylas (Shansha, Áncash) con MMS

In [6]:
xq = cargar_audio(DATA / "shansha_maricela.ogg")[:30 * SR]
print("Whisper (no sabe quechua):", " ".join(s.text.strip() for s in whisper.transcribe(xq, vad_filter=True)[0]))
mms = Transcriptor("mms", idioma_mms="qwh")              # qwh = quechua de Huaylas; quy Ayacucho, quz Cusco, ayr aimara
print("\nMMS [qwh]:", mms.transcribir(xq))

Whisper (no sabe quechua): ¿Y cómo se llama Houti? Houti, Marcilina Figueroa. ¿Y qué plantas de medicina? ¿Y qué plantas? I plantas anorenta. ¿Y cómo se llama Cora? que se sirven inflamación laboral.



MMS [qwh]: matatqi juti marcelina figeroa tapurishqequi que planta medisinacunapaqa i y imataq juticun que planta y lanta juticun auinsham imapaq sirbin que qora que awrinsha sirbin inframasion labani


Whisper inventa castellano; MMS escribe quechua (*imataq*, *imapaq*...). Hay que validarlo con hablantes de la variedad.

## 5 · ¿Quién habla? Transcripción + huellas de voz + clustering

In [7]:
segs = qwen.segmentos(x2, "es")
print(a_texto(diarizar(x2, segs))[:1500])

[00:00:00] H1: Buenos días profesor y compañeros hoy estoy cerca de aquí es por mi casa y voy a hacer una entrevista acerca del fútbol Comunidad de Masas estoy con las señoras Isabel Vasquez

[00:00:11] H2: veis abarrientos

[00:00:13] H1: muy bien voy a comenzar con la primera pregunta y es Qué expectativas tienen ante este partido.

[00:00:22] H3: que que la afición se encuentra motivado para para ganar este partido

[00:00:30] H1: Me puede decir su nombre Isabel Vásquez.

[00:00:33] H2: la activa que digo pienso es que que acá que el Perú va a dar todo por todo todo su corazón va a entregar para que ganemos no para que seamos cam ganaemos este para ir a las olimpiadas al mundial al mundial me puedes decir este su nombre es abarrientos

[00:00:52] H1: Bien, sigo con la siguiente pregunta. eh Qué jugador piensa que va a ser más determinante antes de que diga su respuesta dígame su nombre por favor Isabel Vázquez

[00:01:06] H3: La padula.

[00:01:08] H1: porque

[00:01:11] H3: porque 

## En vivo y app

```bash
uv run python sessions/13-audio-a-texto/app/en_vivo.py           # micrófono → texto en vivo
```
Modelos de streaming abiertos: **Voxtral Realtime** (Mistral, Apache 2.0), Kyutai STT, Moonshine.